In [10]:
import os
import subprocess
import pandas as pd
from pymol import cmd
cmd.feedback("disable", "all", "everything")  # Disables all output, comment this if something doesn't go as expected

In [11]:
def make_pymol_picture(uni, directory, st, png_pictures, alignment_df, detected_pockets):

    # Define some colors
    COLOR_STRUCTURE = 'wheat'
    COLOR_POCKET_SPHERES = 'skyblue'

    # Initialize PyMOL
    cmd.reinitialize()

    # Prettify session 
    cmd.do("set orthoscopic, on")
    cmd.do("set ray_trace_fog, 0")
    cmd.do("set depth_cue, 0")
    cmd.do("set antialias, 4")
    cmd.do("set ray_trace_mode, 1")
    cmd.do("set ray_trace_gain, 0.005")
    cmd.do("bg_color white")
    cmd.do("set spec_reflect, 0")
    cmd.do("set transparency, 0.1")  
    cmd.do("set sphere_scale, 2")
    cmd.do("set internal_gui_width, 400")

    # Load structure
    cmd.load(os.path.join(directory, f"{st}.pdb"), st)

    # Color and visualize
    cmd.color(COLOR_STRUCTURE, st)
    cmd.show("cartoon", st)
    # cmd.show("surface", st)

    # Get a dict mapping pocket number to pocket residues
    pocket_to_residues = alignment_df[(alignment_df['Uniprot AC'] == uni) & (alignment_df['File name'] == f"{st}.pdb")][['Pocket number', 'Pocket residues (chain_resn)']]
    pocket_to_residues = {i:j for i,j in zip(pocket_to_residues['Pocket number'], pocket_to_residues['Pocket residues (chain_resn)'])}

    # Load pockets
    for ptc in sorted(pocket_to_residues):

        # Load pocket
        cmd.load(os.path.join(detected_pockets, uni, st, "pockets", f"pocket_{ptc}.pdb"), f"pocket_{ptc}_{st}")

        cmd.color(COLOR_POCKET_SPHERES, f"pocket_{ptc}_{st}")
        cmd.show("spheres", f"pocket_{ptc}_{st}")

    # Adjust perspective
    cmd.orient()
    cmd.zoom()

    # Save PyMOL PNG screenshot
    cmd.ray(600, 600)
    output_path = os.path.join(png_pictures, f"{st}.png")
    cmd.png(output_path, ray=0, dpi=150)


In [12]:
# Define paths
root = "."
aligned_dir = os.path.abspath(os.path.join(root, "..", "processed", "aligned_structures"))
aligned_relaxed_dir = os.path.abspath(os.path.join(root, "..", "processed", "aligned_relaxed_structures"))
detected_pockets = os.path.abspath(os.path.join(root, "..", "processed", "detected_pockets"))
png_pictures = os.path.abspath(os.path.join(root, "..", "processed", "pymol_pictures"))

# Create dir if necessary
os.makedirs(png_pictures, exist_ok=True)

# Load data
alignment_df = pd.read_csv(os.path.abspath(os.path.join(root, "..", "processed", "pocket_detection_data.csv")))
uniprots = sorted(set(alignment_df["Uniprot AC"]))


# For each uniprot
for uni in uniprots:

    # Specify directory
    directory = os.path.join(aligned_relaxed_dir, uni)

    # Get reference
    reference = [st.replace(".pdb", "") for st in sorted(os.listdir(directory)) if "alphafold2" in st][0]

    # Get all the other structures, but not the reference
    structures = [st.replace(".pdb", "") for st in sorted(os.listdir(directory)) if "alphafold2" not in st]

    for st in [reference] + structures:

        print(f" -------------- Creating PyMOL PNG picture for {st}  --------------  ")

        make_pymol_picture(uni, directory, st, png_pictures, alignment_df, detected_pockets)

 -------------- Creating PyMOL PNG picture for alphafold2_P9WFS9_model_0  --------------  
 -------------- Creating PyMOL PNG picture for alphafold3_P9WFS9_model_0  --------------  
 -------------- Creating PyMOL PNG picture for alphafold3_P9WFS9_model_1  --------------  
 -------------- Creating PyMOL PNG picture for alphafold3_P9WFS9_model_2  --------------  
 -------------- Creating PyMOL PNG picture for alphafold3_P9WFS9_model_3  --------------  
 -------------- Creating PyMOL PNG picture for alphafold3_P9WFS9_model_4  --------------  
 -------------- Creating PyMOL PNG picture for chai1_P9WFS9_model_0  --------------  
 -------------- Creating PyMOL PNG picture for chai1_P9WFS9_model_1  --------------  
 -------------- Creating PyMOL PNG picture for chai1_P9WFS9_model_2  --------------  
 -------------- Creating PyMOL PNG picture for chai1_P9WFS9_model_3  --------------  
 -------------- Creating PyMOL PNG picture for chai1_P9WFS9_model_4  --------------  
 -------------- Creating